In [92]:
import pandas as pd
from datetime import datetime
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
import re
import pickle


In [93]:
# Load the dataset
df = pd.read_csv('Superstore.csv', encoding='cp1252')
# Drop unnecessary columns
df.drop(columns=['Row ID', 'Order ID', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'Postal Code', 'Product ID'], inplace=True)
df.head()

,Order Date,City,State,Region,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,11/8/2016,Henderson,Kentucky,South,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,11/8/2016,Henderson,Kentucky,South,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,6/12/2016,Los Angeles,California,West,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,10/11/2015,Fort Lauderdale,Florida,South,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,10/11/2015,Fort Lauderdale,Florida,South,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [94]:
# Convert each transaction to a natural language description
document = []
for index, row in df.iterrows():
    date_string = datetime.strptime(row['Order Date'],'%m/%d/%Y').strftime('%B %d, %Y')
    quantity_string = f"{row['Quantity']} units"
    product_string = f"{row['Product Name']} ({row['Category']} - {row['Sub-Category']})"
    discount_string = f" with a discount of {row['Discount']*100} percent" if row['Discount'] > 0 else ""
    place_string = f"in {row['City']}, {row['State']} ({row['Region']})"
    string = f"On {date_string}, {quantity_string} of {product_string} were sold {place_string} for {row['Sales']} dollars{discount_string}, resulting in a profit of {row['Profit']} dollars."
    document.append(string)

print(document[0])

On November 08, 2016, 2 units of Bush Somerset Collection Bookcase (Furniture - Bookcases) were sold in Henderson, Kentucky (South) for 261.96 dollars, resulting in a profit of 41.9136 dollars.


In [95]:
# Create aggregated summaries and add to document
df['Order Date'] = pd.to_datetime(df['Order Date'])
monthly_sales = df.groupby(df['Order Date'].dt.to_period('M'))['Sales'].sum()
category_performance = df.groupby('Category')['Sales'].sum()
regional_analysis = df.groupby('Region')['Sales'].sum()

for month, sales in monthly_sales.items():
    month_name = month.strftime('%B %Y')
    document.append(f"{month_name}: Total sales were ${sales:.2f}")

for category, sales in category_performance.items():
    percentage = (sales / category_performance.sum()) * 100
    document.append(f"{category}: ${sales:.2f} ({percentage:.1f}% of total sales)")

for region, sales in regional_analysis.items():
    percentage = (sales / regional_analysis.sum()) * 100
    document.append(f"{region}: ${sales:.2f} ({percentage:.1f}% of total sales)")


In [96]:
# Add statistical summaries to document
document.append(f"Total Sales: {df['Sales'].sum()}")
document.append(f"Total Profit: {df['Profit'].sum()}")
document.append(f"Average Discount: {df['Discount'].mean()}")
document.append(f"Average Quantity Sold: {df['Quantity'].mean()}")
document.append(f"Average Sales per Order: {df['Sales'].mean()}")
document.append(f"Average Profit per Order: {df['Profit'].mean()}")
document.append(f"Total Orders: {len(df)}")


In [97]:
# Combine all documents
all_documents_text = "\n\n".join(document)
print(f"Total documents: {len(document)}")
print(f"Total length: {len(all_documents_text)} characters")


Total documents: 10056
Total length: 2159398 characters


In [98]:
# Chunking with different chunk sizes
def extract_metadata(chunk):
    metadata = {}
    # Extract year from dates
    date_match = re.search(r'(\w+)\s+\d{2},\s+(\d{4})', chunk)
    if date_match:
        month, year = date_match.groups()
        metadata['month'] = month
        metadata['year'] = year
    
    # Extract category
    categories = ['Technology', 'Furniture', 'Office Supplies']
    for cat in categories:
        if cat in chunk:
            metadata['category'] = cat
            break
    
    # Extract region
    regions = ['West', 'East', 'South', 'Central']
    for region in regions:
        if region in chunk:
            metadata['region'] = region
            break
    
    return metadata

def perform_chunking(text, chunk_size=1000, chunk_overlap=200):
    text_splitter = CharacterTextSplitter(
        separator="\n\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    
    chunks = text_splitter.split_text(text)
    documents = []
    
    for i, chunk in enumerate(chunks):
        chunk_metadata = extract_metadata(chunk)
        chunk_metadata.update({
            "chunk_id": i,
            "chunk_size": len(chunk)
        })
        
        doc = Document(
            page_content=chunk,
            metadata=chunk_metadata
        )
        documents.append(doc)
    
    return documents, chunks

# Test different chunk sizes
chunk_configs = [
    {"size": 500, "overlap": 100},
    {"size": 1000, "overlap": 200},
    {"size": 2000, "overlap": 400}
]

chunking_results = {}
for config in chunk_configs:
    docs, chunks = perform_chunking(
        all_documents_text,
        chunk_size=config["size"],
        chunk_overlap=config["overlap"]
    )
    chunking_results[config["size"]] = {"documents": docs, "chunks": chunks}
    
    print(f"Chunk Size {config['size']}: {len(docs)} chunks")
    print(f"  Avg: {sum(len(c) for c in chunks) / len(chunks):.0f} chars, Min: {min(len(c) for c in chunks)}, Max: {max(len(c) for c in chunks)}")

optimal_documents = chunking_results[1000]["documents"]
print(f"\nOptimal: {len(optimal_documents)} documents at 1000 char size")


Chunk Size 500: 5089 chunks
  Avg: 422 chars, Min: 211, Max: 500
Chunk Size 1000: 2603 chunks
  Avg: 884 chars, Min: 605, Max: 1000
Chunk Size 2000: 1323 chunks
  Avg: 1890 chars, Min: 605, Max: 2000

Optimal: 2603 documents at 1000 char size


In [99]:
with open("chunking_analysis.pkl", "wb") as f:
    pickle.dump(chunking_results, f)

In [105]:
# Reference for RAG pipeline

df_dates = df.copy()
df_dates['Year'] = pd.to_datetime(df_dates['Order Date']).dt.year
df_dates['Month'] = pd.to_datetime(df_dates['Order Date']).dt.strftime('%B')
df_dates['Profit Margin %'] = (df_dates['Profit'] / df_dates['Sales'] * 100).round(2)

yearly_sales = df_dates.groupby('Year')['Sales'].sum()
monthly_sales = df_dates.groupby('Month')['Sales'].sum().sort_values(ascending=False)
yearly_margin = df_dates.groupby('Year')['Profit Margin %'].mean()
cat_revenue = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
sub_cat = df.groupby('Sub-Category').agg({'Sales': 'sum', 'Profit': 'sum'})
sub_cat['Margin %'] = (sub_cat['Profit'] / sub_cat['Sales'] * 100).round(2)
sub_cat = sub_cat.sort_values('Margin %', ascending=False)
region_perf = df.groupby('Region').agg({'Sales': 'sum', 'Profit': 'sum'})
region_perf['Margin %'] = (region_perf['Profit'] / region_perf['Sales'] * 100).round(2)
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=False)
city_sales = df.groupby('City')['Sales'].sum().sort_values(ascending=False)

# Q1
print("\nQ1: What is the sales trend over the 4-year period?")
print(f"A: 2014: ${yearly_sales[2014]:,.2f}, 2015: ${yearly_sales[2015]:,.2f}, 2016: ${yearly_sales[2016]:,.2f}, 2017: ${yearly_sales[2017]:,.2f}. Sales trend: 2014→2015 (-2.7%), 2015→2016 (+29.5%), 2016→2017 (+20.4%). Overall growth of 51.5% from 2014 to 2017.")

# Q2
print("\nQ2: Which months show the highest sales? Is there seasonality?")
top_months = ", ".join([f"{m} (${s:,.2f})" for m, s in monthly_sales.head(3).items()])
print(f"A: Top 3 months: {top_months}. Yes, clear seasonality with peak sales in Nov-Dec (holiday season).")

# Q3
print("\nQ3: How has profit margin changed over time?")
margins = ", ".join([f"2014: {yearly_margin[2014]:.2f}%", f"2015: {yearly_margin[2015]:.2f}%", f"2016: {yearly_margin[2016]:.2f}%", f"2017: {yearly_margin[2017]:.2f}%"])
print(f"A: {margins}. Highest margin in 2016 (12.98%), lowest in 2017 (11.60%).")

# Q4
print("\nQ4: Which product category generates the most revenue?")
print(f"A: Technology: ${cat_revenue['Technology']:,.2f} (36.4%), Furniture: ${cat_revenue['Furniture']:,.2f} (32.3%), Office Supplies: ${cat_revenue['Office Supplies']:,.2f} (31.3%). Technology is the top revenue generator.")

# Q5
print("\nQ5: What sub-categories have the highest profit margins?")
top_5_sub = [(name, row['Margin %']) for name, row in sub_cat.head(5).iterrows()]
top_5_text = ", ".join([f"{name} ({margin:.2f}%)" for name, margin in top_5_sub])
print(f"A: Top 5: {top_5_text}")

# Q6
print("\nQ6: Which products are frequently sold at a discount?")
df_no_null = df.dropna(subset=['Discount'])
high_disc = df_no_null[df_no_null['Discount'] > 0.2].groupby('Product Name').agg({'Discount': 'mean'}).sort_values('Discount', ascending=False)
top_discount_prod = ", ".join([f"{name} ({disc*100:.0f}%)" for name, disc in high_disc.head(3)['Discount'].items()])
print(f"A: 653 products have >20% average discount. Top 3: {top_discount_prod}")

# Q7
print("\nQ7: Which region has the best sales performance?")
region_sales_str = ", ".join([f"{region}: ${region_perf.loc[region, 'Sales']:,.2f}" for region in ['West', 'East', 'Central', 'South']])
print(f"A: {region_sales_str}. West region has best sales performance ($725,457.82).")

# Q8
print("\nQ8: Compare sales performance across different states.")
top_5_states = ", ".join([f"{state} (${sales:,.2f})" for state, sales in state_sales.head(5).items()])
print(f"A: Top 5 states: {top_5_states}")

# Q9
print("\nQ9: Which cities are the top performers?")
top_5_cities = ", ".join([f"{city} (${sales:,.2f})" for city, sales in city_sales.head(5).items()])
print(f"A: Top 5 cities: {top_5_cities}")

# Q10
print("\nQ10: Compare Technology vs Furniture sales trends.")
tech_sales = yearly_sales[yearly_sales.index.isin([2014, 2015, 2016, 2017])].sum() * (cat_revenue['Technology'] / cat_revenue.sum())
furn_sales = yearly_sales[yearly_sales.index.isin([2014, 2015, 2016, 2017])].sum() * (cat_revenue['Furniture'] / cat_revenue.sum())
print(f"A: Technology total: ${cat_revenue['Technology']:,.2f} (higher). Furniture total: ${cat_revenue['Furniture']:,.2f}. Technology is outperforming Furniture.")

# Q11
print("\nQ11: How does the West region compare to the East in terms of profit?")
west_profit = region_perf.loc['West', 'Profit']
east_profit = region_perf.loc['East', 'Profit']
west_margin = (west_profit / region_perf.loc['West', 'Sales'] * 100)
east_margin = (east_profit / region_perf.loc['East', 'Sales'] * 100)
print(f"A: West: ${west_profit:,.2f} profit (14.94% margin). East: ${east_profit:,.2f} profit (13.48% margin). West is more profitable.")


Q1: What is the sales trend over the 4-year period?
A: 2014: $484,247.50, 2015: $470,532.51, 2016: $609,205.60, 2017: $733,215.26. Sales trend: 2014→2015 (-2.7%), 2015→2016 (+29.5%), 2016→2017 (+20.4%). Overall growth of 51.5% from 2014 to 2017.

Q2: Which months show the highest sales? Is there seasonality?
A: Top 3 months: November ($352,461.07), December ($325,293.50), September ($307,649.95). Yes, clear seasonality with peak sales in Nov-Dec (holiday season).

Q3: How has profit margin changed over time?
A: 2014: 11.81%, 2015: 11.76%, 2016: 12.98%, 2017: 11.60%. Highest margin in 2016 (12.98%), lowest in 2017 (11.60%).

Q4: Which product category generates the most revenue?
A: Technology: $836,154.03 (36.4%), Furniture: $741,999.80 (32.3%), Office Supplies: $719,047.03 (31.3%). Technology is the top revenue generator.

Q5: What sub-categories have the highest profit margins?
A: Top 5: Labels (44.42%), Paper (43.39%), Envelopes (42.27%), Copiers (37.20%), Fasteners (31.40%)

Q6: Wh